# 02_clean.ipynb：第二部分《数据存储与管理》

这个 notebook 对应作业第二部分，完成以下内容：

- **2.1 目录结构**：自动创建规范目录和必要文件
- **2.2 基础存储方式 A（CSV）**：保存清洗后的股票数据，并生成 `data/combined/combined_data.csv`
- **2.2 进阶方式 B（Parquet）**：额外保存 `data/clean/stock_clean.parquet`
- **2.3 README 记录**：自动把第二部分说明写入 `README.md`

> 使用说明  
> 1. 把本 notebook 放在 `dshw-p01/` 根目录下运行。  
> 2. 它会读取你第一部分已经下载好的数据：`data/stock/`、`data/index/`、`data/macro/`、`data/finance/finance_long.csv`。  
> 3. 本 notebook **不再调用 baostock / akshare**，只处理你本地已有的 CSV 数据。  
> 4. 运行完成后，会自动生成：
>    - `data/clean/stock_clean.csv`
>    - `data/clean/stock_clean.parquet`
>    - `data/combined/combined_data.csv`
>    - `requirements.txt`
>    - `.gitignore`
>    - `report.html`
>    - 更新后的 `README.md`


In [ ]:
import sys
import subprocess
import importlib.util

def ensure_package(pkg_name: str):
    if importlib.util.find_spec(pkg_name) is None:
        print(f"正在安装缺失依赖：{pkg_name}")
        subprocess.check_call([sys.executable, "-m", "pip", "install", pkg_name])

ensure_package("pyarrow")
print("依赖检查完成。")


In [ ]:
from pathlib import Path
import os
import json
import time
import pandas as pd
import pyarrow.parquet as pq
from IPython.display import display

pd.set_option("display.max_columns", 200)

def find_project_root() -> Path:
    candidates = [Path.cwd(), *Path.cwd().parents]
    for p in candidates:
        if (p / "data" / "stock").exists():
            return p
    raise FileNotFoundError("未找到项目根目录。请把 02_clean.ipynb 放在 dshw-p01 根目录下再运行。")

PROJECT_ROOT = find_project_root()

DATA_DIR = PROJECT_ROOT / "data"
STOCK_DIR = DATA_DIR / "stock"
INDEX_DIR = DATA_DIR / "index"
MACRO_DIR = DATA_DIR / "macro"
FINANCE_DIR = DATA_DIR / "finance"
CLEAN_DIR = DATA_DIR / "clean"
COMBINED_DIR = DATA_DIR / "combined"
OUTPUT_DIR = PROJECT_ROOT / "output"

print("项目根目录：", PROJECT_ROOT)


In [ ]:
# 2.1 自动创建目录结构与必要文件

def create_project_structure():
    for folder in [STOCK_DIR, INDEX_DIR, MACRO_DIR, FINANCE_DIR, CLEAN_DIR, COMBINED_DIR, OUTPUT_DIR]:
        os.makedirs(folder, exist_ok=True)

    placeholders = {
        "requirements.txt": "pandas\npyarrow\nbaostock\nakshare\njupyter\n",
        ".gitignore": "__pycache__/\n.ipynb_checkpoints/\n*.pyc\n*.pyo\n.DS_Store\n",
        "report.html": "<!doctype html><html><head><meta charset='utf-8'><title>report</title></head><body><h1>report placeholder</h1></body></html>\n",
    }
    for filename, content in placeholders.items():
        path = PROJECT_ROOT / filename
        if not path.exists():
            path.write_text(content, encoding="utf-8")

    def make_placeholder_notebook(path: Path, title: str):
        if path.exists():
            return
        nb_content = {
            "cells": [
                {
                    "cell_type": "markdown",
                    "metadata": {},
                    "source": [f"# {title}\n\n这是自动创建的占位 notebook。"]
                }
            ],
            "metadata": {
                "kernelspec": {"display_name": "Python 3", "language": "python", "name": "python3"},
                "language_info": {"name": "python"}
            },
            "nbformat": 4,
            "nbformat_minor": 5
        }
        path.write_text(json.dumps(nb_content, ensure_ascii=False, indent=2), encoding="utf-8")

    make_placeholder_notebook(PROJECT_ROOT / "01_download.ipynb", "01_download.ipynb")
    make_placeholder_notebook(PROJECT_ROOT / "03_analysis.ipynb", "03_analysis.ipynb")

create_project_structure()

print("目录结构检查完成。")
print("- clean :", CLEAN_DIR)
print("- combined :", COMBINED_DIR)
print("- output :", OUTPUT_DIR)


## 读取并清洗股票原始数据

这一步会把 `data/stock/` 中的 10 只股票 CSV 合并成一个面板数据，做基本清洗后保存为：

- `data/clean/stock_clean.csv`
- `data/clean/stock_clean.parquet`


In [ ]:
def load_stock_files():
    stock_files = sorted([p for p in STOCK_DIR.glob("stock_*.csv") if p.name != "stock_universe.csv"])
    if not stock_files:
        raise FileNotFoundError("data/stock/ 下没有找到股票 CSV 文件。请先完成第一部分下载。")

    frames = []
    for f in stock_files:
        df = pd.read_csv(f)
        required_cols = {"date", "code", "open", "close", "high", "low", "volume", "amount"}
        if required_cols.issubset(df.columns):
            frames.append(df.copy())

    if not frames:
        raise ValueError("未找到符合格式的股票 CSV 文件。")

    stock = pd.concat(frames, ignore_index=True)
    stock["date"] = pd.to_datetime(stock["date"], errors="coerce")
    stock["code"] = stock["code"].astype(str).str.zfill(6)

    numeric_cols = ["open", "close", "high", "low", "volume", "amount"]
    for col in numeric_cols:
        stock[col] = pd.to_numeric(stock[col], errors="coerce")

    stock = (
        stock.dropna(subset=["date", "code", "close"])
             .drop_duplicates(subset=["date", "code"])
             .sort_values(["code", "date"])
             .reset_index(drop=True)
    )

    stock["year"] = stock["date"].dt.year
    stock["month"] = stock["date"].dt.to_period("M").astype(str)
    stock["stock_ret"] = stock.groupby("code")["close"].pct_change()

    return stock

stock_clean = load_stock_files()

csv_path = CLEAN_DIR / "stock_clean.csv"
parquet_path = CLEAN_DIR / "stock_clean.parquet"

stock_clean.to_csv(csv_path, index=False, encoding="utf-8-sig")
stock_clean.to_parquet(parquet_path, index=False)

print("清洗后股票数据：", stock_clean.shape)
display(stock_clean.head())
print("已保存：")
print("-", csv_path)
print("-", parquet_path)


## 构建综合数据 `combined_data.csv`

这里把以下数据合并为一份综合面板数据：

- 股票日度数据（`stock_clean`）
- 市场指数（按日合并）
- 宏观指标（按月合并）
- 财务指标（按 `code + year` 合并）

最终输出到：

- `data/combined/combined_data.csv`


In [ ]:
def safe_read_csv(path: Path) -> pd.DataFrame:
    return pd.read_csv(path)

def sanitize_colname(text: str) -> str:
    text = str(text).strip().lower()
    mapping = {
        "roe": "roe",
        "资产负债率": "debt_to_asset",
        "cpi同比增速": "cpi_yoy",
        "1年期lpr": "lpr_1y",
        "沪深300": "hs300",
        "上证综指": "sse_index",
    }
    if text in mapping:
        return mapping[text]
    text = text.replace("%", "pct").replace(" ", "_").replace("-", "_").replace("/", "_")
    return text

def load_index_data():
    index_files = sorted(INDEX_DIR.glob("index_*.csv"))
    if not index_files:
        raise FileNotFoundError("data/index/ 下没有找到指数文件。")

    merged = None
    for f in index_files:
        df = safe_read_csv(f)
        if not {"date", "code", "close"}.issubset(df.columns):
            continue

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        code = str(df["code"].iloc[0]).replace(".0", "")
        code = code.zfill(6)
        close_col = f"index_{code}_close"
        ret_col = f"index_{code}_ret"

        out = df[["date", "close"]].copy()
        out["close"] = pd.to_numeric(out["close"], errors="coerce")
        out = out.sort_values("date").drop_duplicates(subset=["date"])
        out[ret_col] = out["close"].pct_change()
        out = out.rename(columns={"close": close_col})

        merged = out if merged is None else merged.merge(out, on="date", how="outer")

    if merged is None:
        raise ValueError("指数文件存在，但没有可用数据。")

    return merged.sort_values("date").reset_index(drop=True)

def load_macro_data():
    macro_files = sorted(MACRO_DIR.glob("macro_*.csv"))
    if not macro_files:
        raise FileNotFoundError("data/macro/ 下没有找到宏观数据文件。")

    merged = None
    for f in macro_files:
        df = safe_read_csv(f)
        if not {"date", "value"}.issubset(df.columns):
            continue

        df["date"] = pd.to_datetime(df["date"], errors="coerce")
        df["value"] = pd.to_numeric(df["value"], errors="coerce")
        df["month"] = df["date"].dt.to_period("M").astype(str)

        if "indicator" in df.columns and df["indicator"].notna().any():
            col = sanitize_colname(df["indicator"].dropna().iloc[0])
        else:
            col = sanitize_colname(f.stem.replace("macro_", ""))

        out = (df.sort_values("date")
                 .groupby("month", as_index=False)["value"].last()
                 .rename(columns={"value": col}))

        merged = out if merged is None else merged.merge(out, on="month", how="outer")

    if merged is None:
        raise ValueError("宏观数据文件存在，但没有可用数据。")

    return merged.sort_values("month").reset_index(drop=True)

def load_finance_wide():
    finance_path = FINANCE_DIR / "finance_long.csv"
    if not finance_path.exists():
        raise FileNotFoundError("缺少 data/finance/finance_long.csv")

    fin = safe_read_csv(finance_path)
    required = {"code", "year", "indicator", "value"}
    if not required.issubset(fin.columns):
        raise ValueError("finance_long.csv 字段不完整，应包含 code, year, indicator, value")

    fin["code"] = fin["code"].astype(str).str.zfill(6)
    fin["year"] = pd.to_numeric(fin["year"], errors="coerce").astype("Int64")
    fin["value"] = pd.to_numeric(fin["value"], errors="coerce")
    fin["indicator"] = fin["indicator"].astype(str)

    fin_wide = (fin.pivot_table(index=["code", "year"], columns="indicator", values="value", aggfunc="first")
                  .reset_index())

    rename_map = {}
    for col in fin_wide.columns:
        rename_map[col] = sanitize_colname(col) if col not in ["code", "year"] else col
    fin_wide = fin_wide.rename(columns=rename_map)

    return fin_wide

index_df = load_index_data()
macro_df = load_macro_data()
finance_wide = load_finance_wide()

combined = (
    stock_clean.merge(index_df, on="date", how="left")
               .merge(macro_df, on="month", how="left")
               .merge(finance_wide, on=["code", "year"], how="left")
               .sort_values(["code", "date"])
               .reset_index(drop=True)
)

combined_path = COMBINED_DIR / "combined_data.csv"
combined.to_csv(combined_path, index=False, encoding="utf-8-sig")

print("综合数据：", combined.shape)
display(combined.head())
print("已保存：", combined_path)


## Parquet 演示：按列读取、查看 Schema、与 CSV 做对比

这一节正对应作业方式 B 的要求。


In [ ]:
# 1) 按列读取（只加载需要的列）
parquet_subset = pd.read_parquet(parquet_path, columns=["date", "code", "close"])
print("Parquet 按列读取结果：", parquet_subset.shape)
display(parquet_subset.head())

# 2) 查看 schema
schema = pq.read_schema(parquet_path)
print("Parquet schema：")
print(schema)


In [ ]:
# 3) 与 CSV 对比：读取速度与文件体积
t0 = time.time()
csv_full = pd.read_csv(csv_path)
csv_time = time.time() - t0
csv_size_kb = csv_path.stat().st_size / 1024

t0 = time.time()
parquet_full = pd.read_parquet(parquet_path)
parquet_time = time.time() - t0
parquet_size_kb = parquet_path.stat().st_size / 1024

comparison = pd.DataFrame({
    "format": ["csv", "parquet"],
    "read_seconds": [csv_time, parquet_time],
    "size_kb": [csv_size_kb, parquet_size_kb],
})

display(comparison)

faster = "Parquet" if parquet_time < csv_time else "CSV"
smaller = "Parquet" if parquet_size_kb < csv_size_kb else "CSV"

print(f"读取更快：{faster}")
print(f"文件更小：{smaller}")


## 自动写入 README（第二部分说明）

这一步会把第二部分的文字说明自动追加/更新到 `README.md`，包括：

- 目录结构
- CSV 的优点与局限
- 为什么选择 Parquet
- 本次实际比较结果
- 对“什么场景下差异更显著”的文字回答


In [ ]:
def build_tree_text():
    return """dshw-p01/
├── README.md
├── report.html
├── requirements.txt
├── .gitignore
├── 01_download.ipynb
├── 02_clean.ipynb
├── 03_analysis.ipynb
├── data/
│   ├── stock/
│   ├── index/
│   ├── macro/
│   ├── finance/
│   ├── clean/
│   │   ├── stock_clean.csv
│   │   └── stock_clean.parquet
│   └── combined/
│       └── combined_data.csv
├── output/
└── download_log.txt"""

def interpret_comparison(csv_time, parquet_time, csv_size_kb, parquet_size_kb):
    speed_gap = abs(csv_time - parquet_time)
    size_gap = abs(csv_size_kb - parquet_size_kb)

    if speed_gap < 0.05:
        speed_text = "在本次数据规模下，CSV 与 Parquet 的读取速度差异不算特别大。"
    else:
        faster = "Parquet" if parquet_time < csv_time else "CSV"
        speed_text = f"在本次数据规模下，{faster} 的读取速度更快，差异已经可以观察到。"

    if size_gap / max(csv_size_kb, parquet_size_kb) < 0.1:
        size_text = "两种格式的文件体积差异不算特别明显。"
    else:
        smaller = "Parquet" if parquet_size_kb < csv_size_kb else "CSV"
        size_text = f"从文件体积看，{smaller} 更节省存储空间。"

    scenario_text = (
        "当数据规模更大、列更多、需要频繁按列读取，或者需要反复进行批量分析时，"
        "Parquet 的优势会更明显；因为它是列式存储，类型信息更完整，通常也有更好的压缩效果。"
        "而 CSV 更适合小规模数据、人工查看、跨软件交换和快速导入导出。"
    )
    return speed_text, size_text, scenario_text

speed_text, size_text, scenario_text = interpret_comparison(csv_time, parquet_time, csv_size_kb, parquet_size_kb)

part2_md = f"""

<!-- part2_start -->
## 第二部分：数据存储与管理

### 2.1 目录结构

本项目按照作业要求，使用 Python 代码自动创建并维护如下目录结构：

```text
{build_tree_text()}
```

其中，所有文件名均使用小写字母和下划线命名，不包含空格与中文。

### 2.2 数据存储格式

#### 方式 A：CSV（必做）

- 所有原始数据继续以 CSV 格式保存；
- 清洗后的股票面板数据保存为 `data/clean/stock_clean.csv`；
- 合并后的综合数据保存为 `data/combined/combined_data.csv`。

CSV 的优点在于：

1. 通用性强，几乎所有数据分析软件都可以直接读取；
2. 可读性高，便于人工检查和调试；
3. 适合课程作业中的小中规模数据交换与提交。

CSV 的局限在于：

1. 不保存严格的数据类型信息，读取时往往需要再次转换类型；
2. 文件体积通常较大；
3. 不支持高效的按列读取，在列很多或数据量较大时性能较弱。

#### 方式 B：Parquet（进阶）

本作业选择 **方式 B：Parquet** 作为进阶格式，并将清洗后的股票数据额外保存为：

- `data/clean/stock_clean.parquet`

选择 Parquet 的理由是：

1. Parquet 是列式存储格式，适合金融面板数据中“按列读取”的分析场景；
2. 它能够保留更明确的 schema（类型契约），减少重复清洗成本；
3. 与 CSV 相比，Parquet 往往在读取效率和压缩体积方面更有优势，便于展示对进阶存储工具的掌握。

### 2.3 Parquet 演示结果

本次在 `02_clean.ipynb` 中完成了以下演示：

- 只读取 Parquet 中的 `date`、`code`、`close` 三列；
- 使用 `pyarrow.parquet.read_schema()` 查看 schema；
- 比较 CSV 与 Parquet 的读取耗时和文件体积。

本次实测结果如下：

- CSV 读取耗时：`{csv_time:.4f}` 秒
- Parquet 读取耗时：`{parquet_time:.4f}` 秒
- CSV 文件大小：`{csv_size_kb:.1f}` KB
- Parquet 文件大小：`{parquet_size_kb:.1f}` KB

文字说明：

- {speed_text}
- {size_text}
- {scenario_text}

<!-- part2_end -->
"""

readme_path = PROJECT_ROOT / "README.md"
existing = readme_path.read_text(encoding="utf-8") if readme_path.exists() else "# dshw-p01\n"

import re
pattern = re.compile(r"<!-- part2_start -->.*?<!-- part2_end -->", re.S)
if pattern.search(existing):
    updated = pattern.sub(part2_md.strip(), existing)
else:
    updated = existing.rstrip() + part2_md

readme_path.write_text(updated, encoding="utf-8")
print("README 已更新：", readme_path)


In [ ]:
print("第二部分全部完成。你现在可以检查以下文件：")
print("-", CLEAN_DIR / "stock_clean.csv")
print("-", CLEAN_DIR / "stock_clean.parquet")
print("-", COMBINED_DIR / "combined_data.csv")
print("-", PROJECT_ROOT / "README.md")
print("-", PROJECT_ROOT / "requirements.txt")
print("-", PROJECT_ROOT / ".gitignore")
print("-", PROJECT_ROOT / "report.html")
